In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Path & Data Loading
PROJECT_ROOT = Path(r"C:\Users\USER\OneDrive\New folder\Desktop\KENYA_HOUSEHOLD_ELECTRIC_POWER_CONSUMPTION")
parquet_path = PROJECT_ROOT / "cleaned_household_power.parquet"

df = pd.read_parquet(parquet_path)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)

df = df.sort_index()

# Identify primary active power column safely
power_col = next(
    (col for col in ["Global_active_power", "global_active_power"] if col in df.columns),
    df.columns[0]
)
df[power_col] = pd.to_numeric(df[power_col], errors="coerce")

# 2. Temporal & Load Regime Features
df["Hour"] = df.index.hour
df["DayOfWeek"] = df.index.dayofweek  # 0=Mon, 6=Sun
df["Weekday_Name"] = df.index.day_name()
df["Month"] = df.index.month
df["Month_Name"] = df.index.strftime("%b")

# Load regimes based on quartiles of active power
df["Load_Regime"] = pd.qcut(
    df[power_col],
    q=4,
    labels=["Q1: Low/Idle", "Q2: Moderate", "Q3: Elevated", "Q4: Peak Draw"]
)

print(f"Loaded shape: {df.shape}")
print(f"Target active power column: '{power_col}'")
df[[power_col, "Hour", "Weekday_Name", "Month_Name", "Load_Regime"]].head()

Loaded shape: (968294, 18)
Target active power column: 'Global_active_power'


,Global_active_power,Hour,Weekday_Name,Month_Name,Load_Regime
DateTime,,,,,
2006-12-16 17:24:00,4.216,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:25:00,5.360,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:26:00,5.374,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:27:00,5.388,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:28:00,3.666,17,Saturday,Dec,Q4: Peak Draw


In [2]:
# --- STEP 2: Central Tendency, Spread, Shape & Robust Summaries ---
if "df" not in globals() or "power_col" not in globals():
    from pathlib import Path
    import pandas as pd

    PROJECT_ROOT = Path(r"C:\Users\USER\OneDrive\New folder\Desktop\KENYA_HOUSEHOLD_ELECTRIC_POWER_CONSUMPTION")
    df = pd.read_parquet(PROJECT_ROOT / "cleaned_household_power.parquet")
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    power_col = next(
        (col for col in ["Global_active_power", "global_active_power"] if col in df.columns),
        df.columns[0]
    )
    df[power_col] = pd.to_numeric(df[power_col], errors="coerce")
s = df[power_col].dropna()

# Central tendency
mean_val = s.mean()
median_val = s.median()
mode_val = s.mode().iloc[0] if not s.mode().empty else np.nan

# Spread & dispersion
min_val = s.min()
max_val = s.max()
range_val = max_val - min_val
q25, q50, q75 = s.quantile([0.25, 0.50, 0.75])
iqr_val = q75 - q25
var_val = s.var(ddof=1)
std_val = s.std(ddof=1)

# Mean Absolute Deviation (around mean and around median)
mad_mean = (s - mean_val).abs().mean()
mad_median = (s - median_val).abs().median()  # True MAD (median absolute deviation)

# Shape & relative variability
skew_val = s.skew()
kurt_excess = s.kurt()          # Fisher (excess kurtosis; normal = 0)
kurt_pearson = kurt_excess + 3  # Pearson kurtosis (normal = 3)
cv_val = std_val / mean_val if mean_val != 0 else np.nan

# Extended quantiles & percentiles
quantiles = s.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

summary_df = pd.DataFrame({
    "Metric": [
        "Count (Valid)", "Mean", "Median", "Mode (1st)",
        "Min", "Max", "Range (Max - Min)",
        "Q1 (25th Percentile)", "Q2 (50th / Median)", "Q3 (75th Percentile)",
        "IQR (Q3 - Q1)", "Variance (Sample)", "Std Deviation (Sample)",
        "MAD (Mean Abs Dev)", "MAD (Median Abs Dev)",
        "Skewness", "Excess Kurtosis", "Pearson Kurtosis",
        "Coeff. of Variation (CV)", "95th Percentile", "99th Percentile"
    ],
    "Value": [
        len(s), mean_val, median_val, mode_val,
        min_val, max_val, range_val,
        q25, q50, q75,
        iqr_val, var_val, std_val,
        mad_mean, mad_median,
        skew_val, kurt_excess, kurt_pearson,
        cv_val, quantiles.loc[0.95], quantiles.loc[0.99]
    ]
})

summary_df["Value"] = summary_df["Value"].round(4)
print(summary_df.to_string(index=False))

                  Metric       Value
           Count (Valid) 967828.0000
                    Mean      1.1744
                  Median      0.6880
              Mode (1st)      0.3220
                     Min      0.0760
                     Max     10.6700
       Range (Max - Min)     10.5940
    Q1 (25th Percentile)      0.3120
      Q2 (50th / Median)      0.6880
    Q3 (75th Percentile)      1.6080
           IQR (Q3 - Q1)      1.2960
       Variance (Sample)      1.3096
  Std Deviation (Sample)      1.1444
      MAD (Mean Abs Dev)      0.8801
    MAD (Median Abs Dev)      0.4880
                Skewness      1.7022
         Excess Kurtosis      3.5745
        Pearson Kurtosis      6.5745
Coeff. of Variation (CV)      0.9745
         95th Percentile      3.5700
         99th Percentile      5.1680


In [4]:
# 1. Conditional summary by Hour of Day
from pathlib import Path
import numpy as np
import pandas as pd

if "df" not in globals() or "power_col" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\USER\OneDrive\New folder\Desktop\KENYA_HOUSEHOLD_ELECTRIC_POWER_CONSUMPTION")
    df = pd.read_parquet(PROJECT_ROOT / "cleaned_household_power.parquet")
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    power_col = next(
        (col for col in ["Global_active_power", "global_active_power"] if col in df.columns),
        df.columns[0]
    )
    df[power_col] = pd.to_numeric(df[power_col], errors="coerce")

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)

if "Hour" not in df.columns:
    df["Hour"] = df.index.hour
if "DayOfWeek" not in df.columns:
    df["DayOfWeek"] = df.index.dayofweek
if "Weekday_Name" not in df.columns:
    df["Weekday_Name"] = df.index.day_name()
if "Month" not in df.columns:
    df["Month"] = df.index.month
if "Month_Name" not in df.columns:
    df["Month_Name"] = df.index.strftime("%b")
if "Load_Regime" not in df.columns:
    df["Load_Regime"] = pd.qcut(
        df[power_col],
        q=4,
        labels=["Q1: Low/Idle", "Q2: Moderate", "Q3: Elevated", "Q4: Peak Draw"]
    )

s = df[power_col].dropna()
hourly_stats = df.groupby("Hour")[power_col].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
    max_peak="max"
).round(3)

print("=== Top 5 Hours by MEAN Active Power ===")
print(hourly_stats.sort_values("mean", ascending=False).head(5))

print("\n=== Top 5 Hours by PEAK (MAX) Active Power ===")
print(hourly_stats.sort_values("max_peak", ascending=False).head(5))

# 2. Conditional summary by Day of Week (ordered Mon -> Sun)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily_stats = df.groupby("Weekday_Name")[power_col].agg(
    mean="mean",
    median="median",
    std="std",
    var="var",
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
    max_peak="max"
).reindex(day_order).round(3)

daily_stats["CV"] = (daily_stats["std"] / daily_stats["mean"]).round(3)
print("\n=== Daily Variability & Central Tendency ===")
print(daily_stats)
print(f"\nHighest Std Dev Day: {daily_stats['std'].idxmax()} ({daily_stats['std'].max()})")
print(f"Highest IQR Day:     {daily_stats['iqr'].idxmax()} ({daily_stats['iqr'].max()})")

# 3. Conditional summary by Month
monthly_stats = df.groupby(["Month", "Month_Name"])[power_col].agg(
    mean="mean",
    median="median",
    std="std",
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
    max_peak="max"
).round(3)
print("\n=== Monthly Summary ===")
print(monthly_stats)

# 4. Conditional summary by Load Regime
regime_stats = df.groupby("Load_Regime", observed=False)[power_col].agg(
    count="count",
    share_pct=lambda x: len(x) / len(s) * 100,
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
).round(3)
print("\n=== Load Regime Summary ===")
print(regime_stats)

# 5. Sub-metering comparison
sub_cols = [c for c in ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"] if c in df.columns]
if sub_cols:
    sub_stats = pd.DataFrame({
        "mean": df[sub_cols].mean(),
        "median": df[sub_cols].median(),
        "std": df[sub_cols].std(),
        "iqr": df[sub_cols].quantile(0.75) - df[sub_cols].quantile(0.25),
        "skew": df[sub_cols].skew(),
        "kurtosis": df[sub_cols].kurt(),
        "max": df[sub_cols].max(),
        "zero_share_pct": (df[sub_cols] == 0).mean() * 100
    }).round(3)
    print("\n=== Sub-Metering Comparison ===")
    print(sub_stats)

=== Top 5 Hours by MEAN Active Power ===
      count   mean  median    std    iqr  max_peak
Hour                                              
21    42208  2.051   1.868  1.385  1.874     9.410
20    42285  2.038   1.776  1.461  2.088    10.348
19    42169  1.883   1.612  1.449  2.118    10.670
7     40873  1.557   1.498  1.128  1.842     9.486
22    42064  1.546   1.330  1.185  1.594     9.272

=== Top 5 Hours by PEAK (MAX) Active Power ===
      count   mean  median    std    iqr  max_peak
Hour                                              
19    42169  1.883   1.612  1.449  2.118    10.670
20    42285  2.038   1.776  1.461  2.088    10.348
1     38581  0.591   0.328  0.681  0.252    10.290
17    40829  1.086   0.582  1.084  1.208     9.590
7     40873  1.557   1.498  1.128  1.842     9.486

=== Daily Variability & Central Tendency ===
               mean  median    std    var    iqr  max_peak     CV
Weekday_Name                                                     
Monday        1.058